# NB1 - Container Supervised Baselines

Trains YOLOv8s and RT-DETR-l on the container dataset (100 epochs), enforces the
T1 pipeline-validity gate, evaluates both on the held-out test split, and pushes
`ausdm-v3-container-supervised` for NB5.

**Kaggle setup required before running:** phone-verified account, internet on,
Roboflow API key stored as Secret `ROBOFLOW_API_KEY`. This notebook rebuilds the
container data itself from Roboflow with the same seed-0 split as NB0, so it does
not depend on NB0 having pushed the image data.

**Runtime ~3.9 h GPU** (YOLOv8s ~50 min, RT-DETR-l ~3 h). Use Save Version -
Save and Run All (Commit), not an interactive session.


## 0. One-time environment setup

In [ ]:
# One-time environment setup. Runs every session; -q keeps output small.
# Same pins as NB0. Uninstall ray to avoid ultralytics 8.3.40's broken raytune callback.
! pip install -q ultralytics==8.3.40 roboflow==1.1.50 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback: pretend `ray` is not installed
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
DATA_ROOT      = KAGGLE_WORKING / 'data'
CONTAINER_DIR  = DATA_ROOT / 'container'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
RESULTS_DIR    = KAGGLE_WORKING / 'results'          # metrics + figures pushed to NB5
STATE_FILE     = KAGGLE_WORKING / 'nb1_state.json'

# Split ratios (plan section 5.1) - identical to NB0 so the test split matches
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.20
TEST_FRAC  = 0.10

# Roboflow project pin (plan section 8a)
RF_WORKSPACE = 'dissertation-t3buk'
RF_PROJECT   = 'container-number-detection'
RF_VERSION   = 3

# Training config (plan section 5.1). Report all of these in the paper.
IMGSZ         = 640
EPOCHS        = 100
BATCH_YOLO    = 16          # YOLOv8s
BATCH_RTDETR  = 8           # RT-DETR-l, avoids OOM on T4 (plan trap)
OPTIMIZER     = 'auto'      # ultralytics default
LR0           = 0.01
LRF           = 0.01
PATIENCE      = 50
DEVICE        = 0

# Model variants (stated explicitly - v2 never did)
YOLO_WEIGHTS   = 'yolov8s.pt'
RTDETR_WEIGHTS = 'rtdetr-l.pt'

# Pre-registered gate (plan section 6). Set before running; do not change after seeing results.
T1_THRESHOLD = 0.60        # YOLOv8s test mAP50-95 must reach this or the matrix halts

# Kaggle dataset this notebook pushes for NB5
DATASET_SLUG = 'ausdm-v3-container-supervised'
# If auto-detect of your Kaggle username fails (NB0 hit 'Invalid Owner Id'),
# set it explicitly here, e.g. KAGGLE_USERNAME_OVERRIDE = ''
KAGGLE_USERNAME_OVERRIDE = ''

for d in [DATA_ROOT, CONTAINER_DIR, RUNS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING} (exists={KAGGLE_WORKING.exists()})')
print(f'RUNS_DIR       = {RUNS_DIR}')
print(f'RESULTS_DIR    = {RESULTS_DIR}')
print(f'T1_THRESHOLD   = {T1_THRESHOLD}  (YOLOv8s test mAP50-95)')
print(f'SEED           = {SEED}')

## 2. Resumable state and restore

A committed run starts from an empty `/kaggle/working`. To resume after an
interruption, mount the previously pushed `ausdm-v3-container-supervised` and this
cell copies its weights, metrics, and state back so completed stages are skipped.


In [ ]:
# _load_state / _mark_done / _is_done - stage-level idempotency across sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

# _restore_from_prior_push - pull a prior NB1 output back into /kaggle/working
def _restore_from_prior_push():
    if STATE_FILE.exists():
        return 'local state already present'
    if not KAGGLE_INPUT.exists():
        return 'no /kaggle/input'
    for cand in KAGGLE_INPUT.rglob('nb1_state.json'):
        src = cand.parent
        shutil.copy2(cand, STATE_FILE)
        for f in ['container_supervised_metrics.csv']:
            if (src / f).exists():
                shutil.copy2(src / f, RESULTS_DIR / f)
        for sub in ['weights', 'figures']:
            if (src / sub).exists():
                shutil.copytree(src / sub, RESULTS_DIR / sub, dirs_exist_ok=True)
        return f'restored from {src}'
    return 'no prior push mounted'

print('restore:', _restore_from_prior_push())
print('current state:', json.dumps(_load_state(), indent=2))

## 3. Secret assertion and Kaggle username - fail before any compute

NB0's push failed with `Invalid Owner Id` because the username resolved to
`unknown`. Detect it robustly here and stop now if it cannot be found, rather
than after 50 minutes of training.


In [ ]:
# _get_roboflow_key - Roboflow Secret, needed to rebuild the container data
def _get_roboflow_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception as e:
        raise RuntimeError(
            'ROBOFLOW_API_KEY Kaggle Secret not found. Add-ons -> Secrets -> add '
            'ROBOFLOW_API_KEY, attach to this notebook, restart the kernel. '
            'Original error: %s' % e
        )

ROBOFLOW_API_KEY = _get_roboflow_key()
assert len(ROBOFLOW_API_KEY) > 10, 'Roboflow key looks too short'

# _kaggle_user - try every credential source Kaggle might expose, in order
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    # 1) authenticated Kaggle API (reads whatever the CLI is using)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u:
            return u
    except Exception:
        pass
    # 2) environment
    u = os.environ.get('KAGGLE_USERNAME')
    if u:
        return u
    # 3) kaggle.json
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            return json.loads(kj.read_text())['username']
        except Exception:
            pass
    # 4) an input mount path of the form /kaggle/input/datasets/<user>/<slug>
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i + 1 < len(parts):
                    return parts[i + 1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print(f'Roboflow key   : OK ({len(ROBOFLOW_API_KEY)} chars)')
print(f'Kaggle username: {KAGGLE_USER}')
if KAGGLE_USER == 'unknown':
    print('WARNING: username not detected. Set KAGGLE_USERNAME_OVERRIDE in section 1 '
          'before the dry-run push, or the push will fail with Invalid Owner Id.')

## 4. Rebuild the container dataset (Roboflow v3, seed-0 re-split)

Same download and re-split code as NB0. The split is deterministic (sorted, then
shuffled with `seed 0`), so the 701 / 200 / 100 train / valid / test partition
here is byte-identical to NB0's. This is what guarantees the reported test-split
numbers use the intended held-out images.


In [ ]:
# _download_container - Roboflow SDK, pinned to v3
def _download_container():
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    proj = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    tgt  = KAGGLE_WORKING / f'rf_raw_v{RF_VERSION}'
    if not tgt.exists():
        proj.version(RF_VERSION).download('yolov8', location=str(tgt))
    return tgt

# _resplit_container - pool all 1001, shuffle with seed 0, re-partition 701/200/100
def _resplit_container(rf_tgt, out_dir):
    if out_dir.exists() and any(out_dir.iterdir()):
        shutil.rmtree(out_dir)
    for split in ['train','valid','test']:
        for sub in ['images','labels']:
            (out_dir / split / sub).mkdir(parents=True, exist_ok=True)
    items = []
    for split in ['train','valid','test']:
        for img in sorted((rf_tgt / split / 'images').iterdir()):
            lab = rf_tgt / split / 'labels' / (img.stem + '.txt')
            items.append((img, lab))
    items.sort(key=lambda t: t[0].name)     # deterministic before shuffle
    random.Random(SEED).shuffle(items)
    n = len(items)
    n_train = int(round(n * TRAIN_FRAC))
    n_val   = int(round(n * VAL_FRAC))
    plan = {'train': items[:n_train],
            'valid': items[n_train:n_train+n_val],
            'test' : items[n_train+n_val:]}
    for split, pairs in plan.items():
        for img, lab in pairs:
            shutil.copy2(img, out_dir / split / 'images' / img.name)
            if lab.exists():
                shutil.copy2(lab, out_dir / split / 'labels' / lab.name)
    return {s: len(v) for s, v in plan.items()}

if not _is_done('container_data'):
    rf_tgt = _download_container()
    total = sum(len(list((rf_tgt / s / 'images').iterdir())) for s in ['train','valid','test'])
    assert total == 1001, f'Expected 1001 images, got {total}'
    counts = _resplit_container(rf_tgt, CONTAINER_DIR)
    assert counts == {'train':701,'valid':200,'test':100}, f'Unexpected split: {counts}'
    print('container re-split counts:', counts)
    _mark_done('container_data', {'counts': counts})
else:
    counts = _load_state()['container_data']['counts']
    print('SKIP container_data, counts:', counts)

# data.yaml with a test key so model.val(split='test') uses the held-out split
CONTAINER_YAML = CONTAINER_DIR / 'data.yaml'
CONTAINER_YAML.write_text(f'''path: {CONTAINER_DIR}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names: ['container-number']
''')
n_test = len(list((CONTAINER_DIR / 'test' / 'images').iterdir()))
print('container data.yaml:', CONTAINER_YAML, '| test images:', n_test)

## 5. Neutralise the raytune callback

ray 3.x removed a symbol the ultralytics 8.3.40 raytune callback still calls.
Same no-op patch as NB0, applied before any `.train()`.


In [ ]:
# Rewrite the callback FUNCTION to a no-op so any cached reference is silent too.
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)

## 6. Authenticated dry-run push

Surface any permission or owner-id problem in under a minute, before committing
~4 hours of GPU. `kaggle datasets create` writes the first version; a re-run
falls back to `kaggle datasets version`.


In [ ]:
# _stage_push_dir - (re)build the staging folder and write metadata
PUSH_DIR = KAGGLE_WORKING / 'push_stage'

def _stage_push_dir(files=None, dirs=None, message='NB1'):
    if PUSH_DIR.exists():
        shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    for f in (files or []):
        if Path(f).exists():
            shutil.copy2(f, PUSH_DIR / Path(f).name)
    for d in (dirs or []):
        if Path(d).exists():
            shutil.copytree(d, PUSH_DIR / Path(d).name, dirs_exist_ok=True)
    meta = {'title': 'AusDM v3 container supervised',
            'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
            'licenses': [{'name': 'CC-BY-SA-4.0'}]}
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
    return meta

# _push - create on first run, version-bump thereafter
def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
assert KAGGLE_USER != 'unknown', (
    'Kaggle username is unknown. Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.')

# marker-only dry run
(RESULTS_DIR / 'nb1_marker.txt').write_text('nb1 dry-run ok\n')
_stage_push_dir(files=[RESULTS_DIR / 'nb1_marker.txt'], message='NB1 dry-run')
if _push('NB1 dry-run'):
    print('dry-run push OK ->', f'{KAGGLE_USER}/{DATASET_SLUG}')
else:
    raise RuntimeError('Dry-run push failed. Fix auth / username before training.')

## 7. Train YOLOv8s (100 epochs, batch 16)

`yolov8s.pt`, imgsz 640, optimizer auto, lr0 0.01, lrf 0.01, patience 50, seed 0.


In [ ]:
# _yolo_best - path to the best checkpoint ultralytics writes
YOLO_RUN  = RUNS_DIR / 'yolov8s_container'
YOLO_BEST = YOLO_RUN / 'weights' / 'best.pt'

if not _is_done('train_yolov8s') and not YOLO_BEST.exists():
    from ultralytics import YOLO
    model = YOLO(YOLO_WEIGHTS)
    model.train(data=str(CONTAINER_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH_YOLO,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='yolov8s_container',
                exist_ok=True, verbose=False, plots=True)
    assert YOLO_BEST.exists(), 'YOLOv8s best.pt not written'
    _mark_done('train_yolov8s', {'best': str(YOLO_BEST)})
    print('YOLOv8s training done ->', YOLO_BEST)
else:
    print('SKIP train_yolov8s, best at', YOLO_BEST)

## 8. T1 gate - YOLOv8s test mAP50-95 must reach 0.60

Pre-registered in the plan. Below the threshold the annotation conversion is
suspect, so the matrix halts here rather than spending three hours on RT-DETR.
The metric is read from the held-out test split, not validation.


In [ ]:
# _eval_test - run test-split validation and pull the four headline numbers
def _eval_test(weights, model_kind, name):
    if model_kind == 'yolo':
        from ultralytics import YOLO; m = YOLO(str(weights))
    else:
        from ultralytics import RTDETR; m = RTDETR(str(weights))
    mt = m.val(data=str(CONTAINER_YAML), split='test', imgsz=IMGSZ,
               device=DEVICE, project=str(RUNS_DIR), name=f'{name}_test',
               exist_ok=True, verbose=False, plots=False)
    return {'model': name,
            'split': 'test',
            'mAP50-95': round(float(mt.box.map),   4),
            'mAP50'   : round(float(mt.box.map50), 4),
            'precision': round(float(mt.box.mp),   4),
            'recall'   : round(float(mt.box.mr),   4)}

yolo_test = _eval_test(YOLO_BEST, 'yolo', 'yolov8s')
print('YOLOv8s test metrics:', json.dumps(yolo_test, indent=2))

# Pre-registered gate - do not soften this comparison after seeing the number
gate_pass = yolo_test['mAP50-95'] >= T1_THRESHOLD
print(f'\nT1 gate: test mAP50-95 = {yolo_test["mAP50-95"]}  vs  threshold {T1_THRESHOLD}  '
      f'-> {"PASS" if gate_pass else "FAIL"}')
_mark_done('eval_yolov8s', {'metrics': yolo_test, 'gate_pass': bool(gate_pass)})

# persist YOLOv8s results immediately (checkpoint before the 3 h RT-DETR run)
def _collect_and_push_yolo():
    (RESULTS_DIR / 'weights').mkdir(exist_ok=True)
    shutil.copy2(YOLO_BEST, RESULTS_DIR / 'weights' / 'yolov8s_best.pt')
    rc = YOLO_RUN / 'results.csv'
    if rc.exists():
        shutil.copy2(rc, RESULTS_DIR / 'yolov8s_results.csv')
    pd.DataFrame([yolo_test]).to_csv(RESULTS_DIR / 'container_supervised_metrics.csv', index=False)
    shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb1_state.json')
    _stage_push_dir(dirs=[RESULTS_DIR], message='NB1 YOLOv8s checkpoint')
    return _push('NB1 YOLOv8s checkpoint')

print('checkpoint push:', 'OK' if _collect_and_push_yolo() else 'FAILED')

assert gate_pass, (
    f'T1 FAILED: YOLOv8s test mAP50-95 {yolo_test["mAP50-95"]} < {T1_THRESHOLD}. '
    'Halt and debug the annotation conversion before training RT-DETR.')

**8.1 Explanation.**

- The gate reads mAP50-95 on the held-out **test** split, which is the number the
  paper reports (addresses R1-W3).
- A pass means the label conversion and training pipeline are sound, so RT-DETR
  can proceed. The plan expects supervised scores in the 0.65 to 0.85 band on
  this harder real-world data.
- YOLOv8s weights and `results.csv` are pushed here, so a crash during RT-DETR
  does not lose the supervised result.


## 9. Train RT-DETR-l (100 epochs, batch 8)

Only runs once the gate passes. Batch 8 to avoid OOM on a 16 GB T4 (plan trap).


In [ ]:
# RT-DETR-l - same schedule as YOLOv8s, smaller batch
RTDETR_RUN  = RUNS_DIR / 'rtdetr_container'
RTDETR_BEST = RTDETR_RUN / 'weights' / 'best.pt'

if not _is_done('train_rtdetr') and not RTDETR_BEST.exists():
    from ultralytics import RTDETR
    model = RTDETR(RTDETR_WEIGHTS)
    model.train(data=str(CONTAINER_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH_RTDETR,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='rtdetr_container',
                exist_ok=True, verbose=False, plots=True)
    assert RTDETR_BEST.exists(), 'RT-DETR-l best.pt not written'
    _mark_done('train_rtdetr', {'best': str(RTDETR_BEST)})
    print('RT-DETR-l training done ->', RTDETR_BEST)
else:
    print('SKIP train_rtdetr, best at', RTDETR_BEST)

## 10. Evaluate RT-DETR-l on the held-out test split

In [ ]:
rtdetr_test = _eval_test(RTDETR_BEST, 'rtdetr', 'rtdetr-l')
print('RT-DETR-l test metrics:', json.dumps(rtdetr_test, indent=2))
_mark_done('eval_rtdetr', {'metrics': rtdetr_test})

## 11. Consolidated test-split metrics

In [ ]:
# Both models, test split, into the CSV NB5 will read for Table 1
metrics = [_load_state()['eval_yolov8s']['metrics'],
           _load_state()['eval_rtdetr']['metrics']]
df_metrics = pd.DataFrame(metrics)[['model','split','mAP50-95','mAP50','precision','recall']]
df_metrics.to_csv(RESULTS_DIR / 'container_supervised_metrics.csv', index=False)
print(df_metrics.to_string(index=False))

**11.1 Explanation.**

- Both rows are the held-out test split, directly comparable.
- RT-DETR-l is expected to edge YOLOv8s on mAP50-95 at a higher training cost.
  Report both numbers as the record; there is no valid v2 comparison (plan 2.2).


## 12. Prediction sanity figures - boxes on test images

Mandatory. Reviewer #2 caught v2 Figure 3 showing images with no boxes. Every
saved figure must have predicted boxes drawn on the correct region.


In [ ]:
# _draw_predictions - run inference on 6 fixed test images, draw boxes, save PNG
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

def _draw_predictions(weights, model_kind, name, n=6, conf=0.25):
    if model_kind == 'yolo':
        from ultralytics import YOLO; m = YOLO(str(weights))
    else:
        from ultralytics import RTDETR; m = RTDETR(str(weights))
    test_imgs = sorted((CONTAINER_DIR / 'test' / 'images').iterdir())[:n]
    for img_path in test_imgs:
        res = m.predict(str(img_path), imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        im = Image.open(img_path).convert('RGB'); dr = ImageDraw.Draw(im)
        for b in res.boxes:
            x1,y1,x2,y2 = [float(v) for v in b.xyxy[0]]
            c = float(b.conf[0])
            dr.rectangle([x1,y1,x2,y2], outline='lime', width=4)
            dr.text((x1, max(0,y1-12)), f'{c:.2f}', fill='lime')
        im.save(FIG_DIR / f'{name}_{img_path.stem}.png')
    return len(test_imgs)

n1 = _draw_predictions(YOLO_BEST,   'yolo',   'pred_yolov8s')
n2 = _draw_predictions(RTDETR_BEST, 'rtdetr', 'pred_rtdetr-l')
print(f'prediction figures: yolov8s x{n1}, rtdetr-l x{n2} -> {FIG_DIR}')

In [ ]:
# Inline preview so the box check happens in the notebook, not just on disk
from IPython.display import Image as IPImg, display
for p in (sorted(FIG_DIR.glob('pred_yolov8s_*'))[:2] +
          sorted(FIG_DIR.glob('pred_rtdetr-l_*'))[:2]):
    print(p.name); display(IPImg(str(p), width=420))

**12.1 Explanation.** Each green box must sit on a container number. A blank
image or a box in the wrong place is the exact defect flagged in v2 Figure 3 and
must be fixed before these figures reach NB5.


## 13. Final push - `ausdm-v3-container-supervised`

In [ ]:
# Stage every artifact NB5 needs and push the final version
(RESULTS_DIR / 'weights').mkdir(exist_ok=True)
shutil.copy2(YOLO_BEST,   RESULTS_DIR / 'weights' / 'yolov8s_best.pt')
shutil.copy2(RTDETR_BEST, RESULTS_DIR / 'weights' / 'rtdetr-l_best.pt')
for run, tag in [(YOLO_RUN,'yolov8s'), (RTDETR_RUN,'rtdetr-l')]:
    rc = run / 'results.csv'
    if rc.exists():
        shutil.copy2(rc, RESULTS_DIR / f'{tag}_results.csv')
_mark_done('final_push')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb1_state.json')

_stage_push_dir(dirs=[RESULTS_DIR], message='NB1 final: YOLOv8s + RT-DETR-l')
print('final push:', 'OK' if _push('NB1 final: YOLOv8s + RT-DETR-l') else 'FAILED')

## 14. Conclusion and handoff to NB5

In [ ]:
print('NB1 state:', json.dumps(_load_state(), indent=2))
print()
print('=== container supervised test-split metrics ===')
print(pd.read_csv(RESULTS_DIR / 'container_supervised_metrics.csv').to_string(index=False))
print()
print('Pushed dataset :', f'{KAGGLE_USER}/{DATASET_SLUG}')
print('Contents       : weights/ (2 best.pt), *_results.csv, '
      'container_supervised_metrics.csv, figures/, nb1_state.json')
print('Next           : NB2 (open-vocab) and NB5 (aggregation) mount this dataset')

**Result.** YOLOv8s and RT-DETR-l are trained for 100 epochs on the container
data and scored on the held-out test split. The **T1** gate at 0.60 mAP50-95
guards the pipeline before the expensive RT-DETR run. The consolidated metrics
CSV, both `best.pt` checkpoints, training curves, and box-drawn prediction
figures are pushed as `ausdm-v3-container-supervised` for NB5.

**Comparability.** Numbers are this study's own record on the public dataset;
there is no valid comparison to the v2 private-dataset table (plan 2.2). Scores
are expected below the near-perfect v2 figures because this data has glare, rust,
night capture, and varied camera angles, which is the intended, stronger result.
